# Pokémon TCG AI Battle：2026-08-07 天梯前排卡组统计

可复现分析：官方日 replay、完整 60 张牌表、胜负结果与官方英/日卡牌映射。

## tl;dr

In [1]:
from pathlib import Path
from html import escape
import csv, json, subprocess
from IPython.display import HTML, Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'analysis' / 'top_ladder_decks.py').exists() and (ROOT / 'top_ladder_decks.py').exists():
    ROOT = ROOT.parent
CONDA_PYTHON = Path('/homes/lzhang/mypath/new/envs/trans/bin/python')
OUTPUT_DIR = ROOT / 'analysis' / 'outputs' / 'top_ladder_2026_08_07'
subprocess.run([str(CONDA_PYTHON), '-s', 'analysis/top_ladder_decks.py', '--workers', '16'], cwd=ROOT, check=True)

def load_csv(name):
    with (OUTPUT_DIR / name).open(encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle))

def show_table(rows, columns, formatters=None, limit=None):
    formatters = formatters or {}
    data = rows if limit is None else rows[:limit]
    head = ''.join(f'<th>{escape(label)}</th>' for _, label in columns)
    body = []
    for row in data:
        cells = []
        for key, _ in columns:
            value = row.get(key, '')
            if key in formatters and value not in ('', None):
                value = formatters[key](value)
            cells.append(f'<td>{escape(str(value))}</td>')
        body.append('<tr>' + ''.join(cells) + '</tr>')
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + ''.join(body) + '</tbody></table>'))

pct = lambda value: f'{float(value):.1%}'
num = lambda value: f'{int(float(value)):,}'
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text(encoding='utf-8'))
lines = [
    f"- **主样本覆盖：** {summary['elite_valid_games']:,}/{summary['elite_target_games']:,} 个高分段目标对局可用，共 {summary['elite_deck_appearances']:,} 个牌表出场、{summary['elite_unique_teams']:,} 支队伍。",
    f"- **高分段口径：** 每日按 `min_score` 排名前 10%，当前门槛为 {summary['elite_min_score_cutoff']:.3f}，保证一局内双方都不低于该分数。",
    f"- **当前第一卡组家族：** {summary['top_archetype']}，使用率 {summary['top_archetype_usage_share']:.1%}，非镜像胜率 {summary['top_archetype_nonmirror_win_rate']:.1%}（n={summary['top_archetype_nonmirror_appearances']:,}）。",
    f"- **先手基线：** 高分段中先手胜率 {summary['first_player_win_rate']:.1%}；卡组胜率应与样本量和对局构成一起看。",
]
display(Markdown('\n'.join(lines)))

{
  "as_of_date": "2026-08-07",
  "previous_date": "2026-08-06",
  "source_latest_create_time": "2026-08-07T23:59:34.649512900",
  "timezone_note": "Kaggle manifest create_time has no timezone offset; timestamps are reported as source time.",
  "elite_definition": "Top 10% of daily matches ranked by min_score, so both players clear the cutoff.",
  "elite_min_score_cutoff": 1079.122914,
  "previous_elite_min_score_cutoff": 1072.177282,
  "elite_target_games": 465,
  "elite_valid_games": 464,
  "elite_deck_appearances": 928,
  "elite_unique_teams": 53,
  "current_manifest_games": 4645,
  "current_valid_game_coverage": 0.9980624327233585,
  "first_player_win_rate": 0.5366379310344828,
  "top_archetype": "Mega Froslass ex / Mega Lopunny ex / Dudunsparce",
  "top_archetype_usage_share": 0.2941810344827586,
  "top_archetype_nonmirror_win_rate": 0.43317972350230416,
  "top_archetype_nonmirror_appearances": 217,
  "top_archetype_previous_usage_share": 0.2041036717062635,
  "top_archetype_usage

- **主样本覆盖：** 464/465 个高分段目标对局可用，共 928 个牌表出场、53 支队伍。
- **高分段口径：** 每日按 `min_score` 排名前 10%，当前门槛为 1079.123，保证一局内双方都不低于该分数。
- **当前第一卡组家族：** Mega Froslass ex / Mega Lopunny ex / Dudunsparce，使用率 29.4%，非镜像胜率 43.3%（n=217）。
- **先手基线：** 高分段中先手胜率 53.7%；卡组胜率应与样本量和对局构成一起看。

## Context & Methods

### Key Assumptions

- 把“天梯靠前排”定义为每日按 `min_score` 排名前 10% 的对局。`min_score` 是两名玩家中较低者的分数，因此能保证双方都达到门槛；日清单不提供分数到具体玩家的映射。
- replay 在开局动作中包含双方完整 60 张卡 ID；最终 `rewards` 提供胜负。只把双方牌表完整且奖励有效的对局纳入主分析。
- 卡组家族基于 Pokémon 卡多重集合聚类：对 Pokémon 核心做 IDF 加权 Jaccard，相似度至少 0.55 归为同一家族；Trainer/Energy 的差别作为具体牌表变体。
- 胜率的 Wilson 区间是描述性的；同一队伍的重复对局并非独立样本。镜像对局单独排除后计算主胜率。
- manifest 时间戳不带时区偏移，本文只按源时间使用，不推断时区。

## Data

In [2]:
quality = load_csv('data_quality.csv')
show_table(quality, [
    ('date','日期'), ('manifest_games','Manifest 对局'), ('json_files_present','JSON 文件'),
    ('parsed_games','解析成功'), ('deck_complete_games','完整牌表'),
    ('valid_games','有效对局'), ('valid_game_coverage','有效覆盖率')
], {'manifest_games':num,'json_files_present':num,'parsed_games':num,'deck_complete_games':num,'valid_games':num,'valid_game_coverage':pct})
display(Markdown(f"输出目录：`{OUTPUT_DIR}`"))

日期,Manifest 对局,JSON 文件,解析成功,完整牌表,有效对局,有效覆盖率
2026-08-06,"4,633","4,631","4,631","4,631","4,629",99.9%
2026-08-07,"4,645","4,639","4,639","4,639","4,636",99.8%


输出目录：`/homes/lzhang/pocketmon/analysis/outputs/top_ladder_2026_08_07`

## Results

下面先看卡组家族的出场率与非镜像胜率，再看与前一日的使用率变化。

In [3]:
archetypes = load_csv('archetype_summary.csv')
show_table(archetypes, [
    ('rank','排名'), ('archetype_label','卡组家族'), ('appearances','出场'),
    ('usage_share','使用率'), ('nonmirror_appearances','非镜像 n'),
    ('nonmirror_win_rate','非镜像胜率'), ('nonmirror_ci_low','95% CI 下限'),
    ('nonmirror_ci_high','95% CI 上限'), ('unique_teams','队伍'), ('exact_deck_variants','完整牌表变体')
], {
    'rank':num,'appearances':num,'usage_share':pct,'nonmirror_appearances':num,
    'nonmirror_win_rate':pct,'nonmirror_ci_low':pct,'nonmirror_ci_high':pct,
    'unique_teams':num,'exact_deck_variants':num
}, limit=15)

排名,卡组家族,出场,使用率,非镜像 n,非镜像胜率,95% CI 下限,95% CI 上限,队伍,完整牌表变体
1,Mega Froslass ex / Mega Lopunny ex / Dudunsparce,273,29.4%,217,43.3%,36.9%,50.0%,14,5
2,Marnie's Grimmsnarl ex / Marnie's Morgrem / Froslass,189,20.4%,143,42.0%,34.2%,50.2%,12,3
3,Fezandipiti ex / Alakazam / Kadabra,133,14.3%,107,53.3%,43.9%,62.4%,10,3
4,Mega Lucario ex / Hariyama / Solrock,91,9.8%,87,54.0%,43.6%,64.1%,5,1
5,Teal Mask Ogerpon ex / Mega Kangaskhan ex / Raging Bolt ex,52,5.6%,52,44.2%,31.6%,57.7%,1,1
6,Dragapult ex / Meowth ex / Fezandipiti ex,47,5.1%,47,48.9%,35.3%,62.8%,4,3
7,N’s Zoroark ex / N’s Zorua / N’s Reshiram,35,3.8%,35,71.4%,54.9%,83.7%,1,1
8,Thwackey / Dipplin / Seaking,30,3.2%,30,63.3%,45.5%,78.1%,2,2
9,Mega Kangaskhan ex / Latias ex / Meowth ex,26,2.8%,26,80.8%,62.1%,91.5%,2,1
10,Hydrapple ex / Teal Mask Ogerpon ex / Meowth ex,25,2.7%,25,56.0%,37.1%,73.3%,2,2


### 日对日卡组占比变化

In [4]:
comparison = load_csv('day_comparison.csv')
show_table(comparison, [
    ('archetype_label','卡组家族'), ('current_appearances','8/7 出场'),
    ('current_usage_share','8/7 使用率'), ('previous_appearances','8/6 出场'),
    ('previous_usage_share','8/6 使用率'), ('usage_share_delta_pp','变化 pp')
], {
    'current_appearances':num,'current_usage_share':pct,'previous_appearances':num,
    'previous_usage_share':pct,'usage_share_delta_pp':lambda x:f'{float(x):+.1f}'
}, limit=15)

卡组家族,8/7 出场,8/7 使用率,8/6 出场,8/6 使用率,变化 pp
Mega Froslass ex / Mega Lopunny ex / Dudunsparce,273,29.4%,189,20.4%,+9.0
Marnie's Grimmsnarl ex / Marnie's Morgrem / Froslass,189,20.4%,258,27.9%,-7.5
Fezandipiti ex / Alakazam / Kadabra,133,14.3%,153,16.5%,-2.2
Mega Lucario ex / Hariyama / Solrock,91,9.8%,71,7.7%,+2.1
Teal Mask Ogerpon ex / Mega Kangaskhan ex / Raging Bolt ex,52,5.6%,59,6.4%,-0.8
Dragapult ex / Meowth ex / Fezandipiti ex,47,5.1%,50,5.4%,-0.3
N’s Zoroark ex / N’s Zorua / N’s Reshiram,35,3.8%,0,0.0%,+3.8
Thwackey / Dipplin / Seaking,30,3.2%,1,0.1%,+3.1
Mega Kangaskhan ex / Latias ex / Meowth ex,26,2.8%,43,4.6%,-1.8
Hydrapple ex / Teal Mask Ogerpon ex / Meowth ex,25,2.7%,24,2.6%,+0.1


### 高分段常见卡与相对全场提升

In [5]:
usage = load_csv('card_usage.csv')
for group in ['Pokémon','Trainer','Energy']:
    display(Markdown(f'#### {group}'))
    rows = [row for row in usage if row['card_group'] == group][:12]
    show_table(rows, [
        ('card_id','ID'),('card_name_en','English'),('card_name_jp','日本語'),
        ('elite_inclusion_count','纳入牌表'),('elite_inclusion_rate','高分段纳入率'),
        ('avg_copies_when_included','纳入时均张数'),('field_inclusion_rate','全场纳入率'),
        ('elite_vs_field_lift','Lift')
    ], {
        'card_id':num,'elite_inclusion_count':num,'elite_inclusion_rate':pct,
        'avg_copies_when_included':lambda x:f'{float(x):.2f}',
        'field_inclusion_rate':pct,'elite_vs_field_lift':lambda x:f'{float(x):.2f}x'
    })

#### Pokémon

ID,English,日本語,纳入牌表,高分段纳入率,纳入时均张数,全场纳入率,Lift
860,Snorunt,ユキワラシ,426,45.9%,2.00,42.8%,1.07x
66,Dudunsparce,ノココッチ,410,44.2%,2.70,33.8%,1.31x
305,Dunsparce,ノコッチ,406,43.8%,3.67,33.2%,1.32x
140,Fezandipiti ex,キチキギスex,286,30.8%,1.00,31.1%,0.99x
112,Munkidori,マシマシラ,273,29.4%,3.38,39.9%,0.74x
848,Buneary,ミミロル,273,29.4%,2.18,15.7%,1.88x
849,Mega Lopunny ex,メガミミロップex,273,29.4%,2.13,15.8%,1.86x
174,Fan Rotom,スピンロトム,273,29.4%,1.00,14.9%,1.99x
861,Mega Froslass ex,メガユキメノコex,237,25.5%,2.00,10.9%,2.34x
646,Marnie's Impidimp,マリィのベロバー,189,20.4%,4.00,31.9%,0.64x


#### Trainer

ID,English,日本語,纳入牌表,高分段纳入率,纳入时均张数,全场纳入率,Lift
"1,182",Boss’s Orders,ボスの指令,902,97.2%,2.27,98.2%,0.99x
"1,152",Poké Pad,ポケパッド,851,91.7%,3.85,85.8%,1.07x
"1,227",Lillie's Determination,リーリエの決心,743,80.1%,4.00,80.8%,0.99x
"1,086",Buddy-Buddy Poffin,なかよしポフィン,730,78.7%,3.98,85.4%,0.92x
"1,097",Night Stretcher,夜のタンカ,561,60.5%,2.11,67.7%,0.89x
"1,121",Ultra Ball,ハイパーボール,559,60.2%,3.88,36.1%,1.67x
"1,225",Hilda,トウコ,454,48.9%,3.23,44.8%,1.09x
"1,122",Pokégear 3.0,ポケギア3.0,446,48.1%,1.83,58.3%,0.83x
"1,229",Wally's Compassion,ミツルの思いやり,364,39.2%,3.50,19.5%,2.01x
"1,231",Dawn,ヒカリ,345,37.2%,2.16,58.6%,0.64x


#### Energy

ID,English,日本語,纳入牌表,高分段纳入率,纳入时均张数,全场纳入率,Lift
13,Enriching Energy,リッチエネルギー,406,43.8%,1.00,32.8%,1.34x
11,Mist Energy,ミストエネルギー,294,31.7%,4.00,23.6%,1.35x
3,Basic {W} Energy,基本【水】エネルギー,289,31.1%,2.64,12.5%,2.49x
7,Basic {D} Energy,基本【悪】エネルギー,273,29.4%,8.45,40.0%,0.74x
5,Basic {P} Energy,基本【超】エネルギー,260,28.0%,2.57,26.9%,1.05x
19,Telepath Psychic Energy,テレパス【超】エネルギー,173,18.6%,3.92,18.8%,1.00x
6,Basic {F} Energy,基本【闘】エネルギー,143,15.4%,9.00,6.6%,2.34x
1,Basic {G} Energy,基本【草】エネルギー,124,13.4%,9.27,20.2%,0.67x
4,Basic {L} Energy,基本【雷】エネルギー,60,6.5%,2.13,1.7%,3.81x
2,Basic {R} Energy,基本【炎】エネルギー,49,5.3%,3.86,7.0%,0.77x


### 头部卡组对局矩阵

In [6]:
matrix_rows = load_csv('matchup_win_rate_matrix.csv')
if matrix_rows:
    first_key = next(iter(matrix_rows[0]))
    matrix_columns = [(first_key,'卡组')] + [(key,key) for key in matrix_rows[0] if key != first_key]
    matrix_formatters = {key:pct for key in matrix_rows[0] if key != first_key}
    show_table(matrix_rows, matrix_columns, matrix_formatters)
display(Markdown('对应样本量保存在 `matchup_sample_size_matrix.csv`；空白或小样本对局不应据此下确定性结论。'))

卡组,Dragapult ex / Meowth ex / Fezandipiti ex,Fezandipiti ex / Alakazam / Kadabra,Marnie's Grimmsnarl ex / Marnie's Morgrem / Froslass,Mega Froslass ex / Mega Lopunny ex / Dudunsparce,Mega Lucario ex / Hariyama / Solrock,N’s Zoroark ex / N’s Zorua / N’s Reshiram,Teal Mask Ogerpon ex / Mega Kangaskhan ex / Raging Bolt ex,Thwackey / Dipplin / Seaking
Dragapult ex / Meowth ex / Fezandipiti ex,,42.9%,44.4%,52.4%,33.3%,,50.0%,100.0%
Fezandipiti ex / Alakazam / Kadabra,57.1%,50.0%,44.4%,53.8%,55.6%,83.3%,58.3%,37.5%
Marnie's Grimmsnarl ex / Marnie's Morgrem / Froslass,55.6%,55.6%,50.0%,41.3%,47.4%,16.7%,30.0%,42.9%
Mega Froslass ex / Mega Lopunny ex / Dudunsparce,47.6%,46.2%,58.7%,50.0%,28.6%,7.7%,50.0%,27.3%
Mega Lucario ex / Hariyama / Solrock,66.7%,44.4%,52.6%,71.4%,50.0%,66.7%,75.0%,
N’s Zoroark ex / N’s Zorua / N’s Reshiram,,16.7%,83.3%,92.3%,33.3%,,75.0%,
Teal Mask Ogerpon ex / Mega Kangaskhan ex / Raging Bolt ex,50.0%,41.7%,70.0%,50.0%,25.0%,25.0%,,50.0%
Thwackey / Dipplin / Seaking,0.0%,62.5%,57.1%,72.7%,,,50.0%,


对应样本量保存在 `matchup_sample_size_matrix.csv`；空白或小样本对局不应据此下确定性结论。

### 代表性完整牌表

In [7]:
decklists = load_csv('representative_decklists.csv')
show_table(decklists, [
    ('archetype_rank','家族排名'),('archetype_label','卡组家族'),('card_id','卡 ID'),
    ('card_name_en','English'),('card_name_jp','日本語'),('card_group','类别'),('count','张数')
], {'archetype_rank':num,'card_id':num,'count':num}, limit=80)
display(Markdown('完整前十卡组代表牌表已导出到 `representative_decklists.csv`。'))

家族排名,卡组家族,卡 ID,English,日本語,类别,张数
1,Mega Froslass ex / Mega Lopunny ex / Dudunsparce,66,Dudunsparce,ノココッチ,Pokémon,3
1,Mega Froslass ex / Mega Lopunny ex / Dudunsparce,174,Fan Rotom,スピンロトム,Pokémon,1
1,Mega Froslass ex / Mega Lopunny ex / Dudunsparce,305,Dunsparce,ノコッチ,Pokémon,4
1,Mega Froslass ex / Mega Lopunny ex / Dudunsparce,848,Buneary,ミミロル,Pokémon,2
1,Mega Froslass ex / Mega Lopunny ex / Dudunsparce,849,Mega Lopunny ex,メガミミロップex,Pokémon,2
1,Mega Froslass ex / Mega Lopunny ex / Dudunsparce,860,Snorunt,ユキワラシ,Pokémon,2
1,Mega Froslass ex / Mega Lopunny ex / Dudunsparce,861,Mega Froslass ex,メガユキメノコex,Pokémon,2
1,Mega Froslass ex / Mega Lopunny ex / Dudunsparce,"1,086",Buddy-Buddy Poffin,なかよしポフィン,Trainer,4
1,Mega Froslass ex / Mega Lopunny ex / Dudunsparce,"1,087",Hand Trimmer,ハンドトリマー,Trainer,3
1,Mega Froslass ex / Mega Lopunny ex / Dudunsparce,"1,121",Ultra Ball,ハイパーボール,Trainer,4


完整前十卡组代表牌表已导出到 `representative_decklists.csv`。

## Takeaways

In [8]:
top = archetypes[0]
second = archetypes[1]
delta = float(comparison[0]['usage_share_delta_pp'])
takeaways = [
    f"1. **备战优先级先按出场率排：** `{top['archetype_label']}` 是当前最常见家族（{float(top['usage_share']):.1%}），其次是 `{second['archetype_label']}`（{float(second['usage_share']):.1%}）。",
    f"2. **胜率只作方向性证据：** 头部家族的非镜像胜率为 {float(top['nonmirror_win_rate']):.1%}，95% Wilson 区间 {float(top['nonmirror_ci_low']):.1%}–{float(top['nonmirror_ci_high']):.1%}；同队重复对局和 matchup mix 会影响估计。",
    f"3. **关注结构变化：** 第一家族较前一日使用率变化 {delta:+.1f} 个百分点；赛前应继续按同一口径刷新最新日 replay。",
    "4. **落地建议：** 先针对出场率最高的三类做定向 matchup 回放和起手/先后手切分，再决定是否因小样本高胜率卡组调整构筑。",
]
display(Markdown('\n'.join(takeaways)))

1. **备战优先级先按出场率排：** `Mega Froslass ex / Mega Lopunny ex / Dudunsparce` 是当前最常见家族（29.4%），其次是 `Marnie's Grimmsnarl ex / Marnie's Morgrem / Froslass`（20.4%）。
2. **胜率只作方向性证据：** 头部家族的非镜像胜率为 43.3%，95% Wilson 区间 36.9%–50.0%；同队重复对局和 matchup mix 会影响估计。
3. **关注结构变化：** 第一家族较前一日使用率变化 +9.0 个百分点；赛前应继续按同一口径刷新最新日 replay。
4. **落地建议：** 先针对出场率最高的三类做定向 matchup 回放和起手/先后手切分，再决定是否因小样本高胜率卡组调整构筑。